# Heartland Character Network Graph

A graph to visualise the appearances of the characters on Heartland

## Graph Design

**Nodes**: Represent characters. Sized and colour determined by episode count.

**Edges**: Represent characters who appeared in the same episode. Thickness equates to shared episode count.

**Filter**: by season using the dropdown on the html page. Default is entire series.

![graph screenshot for season 1](img/ui.png)

## Data Gathering

**First run:** Data fetching scrapes 19 season pages + ~280 guestcast API calls from [TV Maze](https://www.tvmaze.com/shows/3164/heartland)

**Subsequent runs:** instantly loads from local cache.

> **_NOTE:_** There was no guest cast data on TV Maze for seasons 2-13


## 1. Setup

In [ ]:
import requests
import json
import time
from pathlib import Path
from bs4 import BeautifulSoup
from itertools import combinations
from collections import defaultdict

import pandas as pd
import networkx as nx
from pyvis.network import Network
import ipywidgets as widgets
from IPython.display import display, HTML
from tqdm.notebook import tqdm

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
GUESTCAST_DIR = DATA_DIR / "guestcast"
SEASON_CAST_DIR = DATA_DIR / "season_cast"
for d in [DATA_DIR, GUESTCAST_DIR, SEASON_CAST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36", ## Makes Maze TV servers think the reqs are from a real browser, not a bot
    "Accept-Language": "en-US,en;q=0.9",
})

print("Setup complete.")

## 2. Fetch Data

Scripts priortise local data if present

### Fetch Season and Episode Data

In [ ]:
def fetch_json(url, cache_path, delay=0.25):
    """Fetch JSON from URL, using local cache if available."""
    if cache_path.exists():
        return json.loads(cache_path.read_text())
    time.sleep(delay)
    resp = SESSION.get(url, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    cache_path.write_text(json.dumps(data))
    return data


def fetch_html(url, cache_path, delay=1.0):
    """Fetch HTML from URL, using local cache if available."""
    if cache_path.exists():
        return cache_path.read_text()
    time.sleep(delay)
    resp = SESSION.get(url, timeout=15)
    resp.raise_for_status()
    cache_path.write_text(resp.text)
    return resp.text

print("Fetching episode list...")
episodes = fetch_json(
    "https://api.tvmaze.com/shows/3164/episodes",
    DATA_DIR / "episodes.json"
)
print(f"  {len(episodes)} episodes across {max(e['season'] for e in episodes)} seasons")

print("Fetching season list...")
seasons_meta = fetch_json(
    "https://api.tvmaze.com/shows/3164/seasons",
    DATA_DIR / "seasons.json"
)

season_id_map = {s["number"]: s["id"] for s in seasons_meta}
print(f"  {len(season_id_map)} seasons found")

### Fetch Cast Data

In [ ]:
print("Scraping regular cast per season (1s delay between requests)...")

def scrape_season_cast(season_number, season_id):
    """Scrape regular cast character names from a TVMaze season page."""
    cache = SEASON_CAST_DIR / f"{season_number}.json"
    if cache.exists():
        return json.loads(cache.read_text())
    
    url = f"https://www.tvmaze.com/seasons/{season_id}/heartland-season-{season_number}"
    html = fetch_html(url, SEASON_CAST_DIR / f"{season_number}.html", delay=1.0)
    soup = BeautifulSoup(html, "html.parser")
    
    # Character names are in <a href="/characters/..."> links
    characters = []
    for a in soup.select('a[href*="/characters/"]'):
        name = a.get_text(strip=True)
        if name:
            characters.append(name)
    
    # Deduplicate while preserving order
    seen = set()
    unique_chars = []
    for c in characters:
        key = c.lower()
        if key not in seen:
            seen.add(key)
            unique_chars.append(c)
    
    cache.write_text(json.dumps(unique_chars))
    return unique_chars


season_regular_cast = {}  # season_number -> [character_name, ...]
for season_num in tqdm(sorted(season_id_map.keys()), desc="Seasons"):
    chars = scrape_season_cast(season_num, season_id_map[season_num])
    season_regular_cast[season_num] = chars

print("\nRegular cast counts per season:")
for s, chars in sorted(season_regular_cast.items()):
    print(f"  Season {s:2d}: {len(chars)} regulars")

In [ ]:
print("Fetching guest cast per episode (0.25s delay)...")

episode_guestcast = {}  # episode_id -> [character_name, ...]
missing = [e for e in episodes if not (GUESTCAST_DIR / f"{e['id']}.json").exists()]
print(f"  {len(episodes) - len(missing)} already cached, fetching {len(missing)} new...")

for ep in tqdm(episodes, desc="Episodes"):
    ep_id = ep["id"]
    data = fetch_json(
        f"https://api.tvmaze.com/episodes/{ep_id}/guestcast",
        GUESTCAST_DIR / f"{ep_id}.json",
        delay=0.25
    )
    episode_guestcast[ep_id] = [entry["character"]["name"] for entry in data if entry.get("character", {}).get("name")]

print(f"Done. Guest cast fetched for {len(episode_guestcast)} episodes.")

guest_by_season = defaultdict(int)
for ep in episodes:
    guest_by_season[ep["season"]] += len(episode_guestcast.get(ep["id"], []))
print("\nGuest cast counts per season:")
for s in sorted(guest_by_season):
    print(f"  Season {s:2d}: {guest_by_season[s]} guests")

## 3. Process Data

In [ ]:
def normalize(name):
    """Normalize character name for deduplication."""
    return name.strip().lower()

canonical_name = {}

def get_canonical(name):
    key = normalize(name)
    if key not in canonical_name:
        canonical_name[key] = name.strip()
    return canonical_name[key]


# Pre-populate canonical names from season cast (regulars first for consistent casing)
for chars in season_regular_cast.values():
    for c in chars:
        get_canonical(c)

ep_meta = {e["id"]: e for e in episodes}

episode_characters = {}  # episode_id -> set of canonical character names

for ep in episodes:
    ep_id = ep["id"]
    season = ep["season"]
    chars = set()
    
    for c in season_regular_cast.get(season, []):
        chars.add(get_canonical(c))
    
    for c in episode_guestcast.get(ep_id, []):
        chars.add(get_canonical(c))
    
    episode_characters[ep_id] = chars


node_episodes = defaultdict(set)  # character -> {episode_id, ...}
for ep_id, chars in episode_characters.items():
    for c in chars:
        node_episodes[c].add(ep_id)


edge_episodes = defaultdict(set)  # frozenset({a, b}) -> {episode_id, ...}
for ep_id, chars in episode_characters.items():
    char_list = sorted(chars)
    for a, b in combinations(char_list, 2):
        edge_episodes[frozenset([a, b])].add(ep_id) # frozenset ensures the set cannot be amended


all_seasons = sorted(set(e["season"] for e in episodes))
print(f"Characters Total: {len(node_episodes)}")
print(f"Character pairs (edges) Total: {len(edge_episodes)}")

In [ ]:
import webbrowser
import threading
import http.server

_server_port = None

def _ensure_server():
    global _server_port
    if _server_port == 3000:
        return 3000
    project_root = str(Path(".").resolve())
    class Handler(http.server.SimpleHTTPRequestHandler):
        def __init__(self, *args, **kwargs):
            super().__init__(*args, directory=project_root, **kwargs)
        def log_message(self, *args): pass
    try:
        httpd = http.server.HTTPServer(("localhost", 3000), Handler)
        threading.Thread(target=httpd.serve_forever, daemon=True).start()
    except OSError:
        pass  # already bound to 3000 from a previous run
    _server_port = 3000
    return 3000


def build_browser_graph():
    season_options = [("All seasons", None)] + [(f"Season {s}", s) for s in all_seasons]
    all_data = {}

    for label, season in season_options:
        scope_ids = (
            set(ep_meta.keys())
            if season is None
            else {ep_id for ep_id, meta in ep_meta.items() if meta["season"] == season}
        )

        node_weights = {}
        for char, ep_ids in node_episodes.items():
            count = len(ep_ids & scope_ids)
            if count > 0:
                node_weights[char] = count

        edge_weights = {}
        for pair, ep_ids in edge_episodes.items():
            count = len(ep_ids & scope_ids)
            if count > 0:
                a, b = list(pair)
                if a in node_weights and b in node_weights:
                    edge_weights[(a, b)] = count

        max_ep = max(node_weights.values(), default=1)
        max_sh = max(edge_weights.values(), default=1)

        nodes = []
        for char, count in node_weights.items():
            total_appearances = count / max_ep
            color = f"#{255:02x}{int(216 - total_appearances * 96):02x}{int(166 - total_appearances * 115):02x}"
            nodes.append({
                "id": char, "label": char,
                "size": 8 + total_appearances * 42,
                "image": "/img/heartland_logo.png",
                "color": {
                    "border": color,
                    "highlight": {"background": "#33baff", "border": "#33baff"},
                    "hover": {"background": "#33baff", "border": "#33baff"}
                },
                "title": f"{char} - {count} episode{'s' if count != 1 else ''}",
            })

        edges = []
        for (a, b), count in edge_weights.items():
            total_appearances = count / max_sh
            alpha = (60 + int(total_appearances * 195)) / 255
            edges.append({
                "from": a, "to": b,
                "width": 0.5 + total_appearances * 9,
                "color": {
                    "color": f"rgba(180,180,255,{alpha:.2f})",
                    "inherit": False,
                    "highlight": "#7833ff",
                    "hover": "#7833ff"},
                "title": f"{count} shared episode{'s' if count != 1 else ''}",
            })

        regulars = (
            [get_canonical(c) for c in season_regular_cast.get(season, [])]
            if season is not None else []
        )

        all_data[label] = {"nodes": nodes, "edges": edges, "regulars": regulars}
        print(f"  {label}: {len(nodes)} characters, {len(edges)} edges")

    options_html = "\n".join(
        f'        <option value="{label}">{label}</option>'
        for label, _ in season_options
    )

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Heartland Character Network</title>
  <script src="https://unpkg.com/vis-network@9.1.9/standalone/umd/vis-network.min.js"></script>
  <style>
    * {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{ background: #1a0f1a; color: #e0e0e0; font-family: system-ui, sans-serif;
            display: flex; flex-direction: column; height: 100vh; overflow: hidden; }}
    header {{ padding: 12px 20px; display: flex; align-items: center; gap: 16px;
              background: #2C1A2E; border-bottom: 1px solid #2e2e4e; flex-shrink: 0; }}
    h1 {{ font-size: 1.1rem; font-weight: 600; letter-spacing: 0.02em; }}
    select {{ background: #2a2a3e; color: #e0e0e0; border: 1px solid #555;
              border-radius: 6px; padding: 6px 12px; font-size: 0.9rem; cursor: pointer; }}
    #regular-toggle {{ display: none; align-items: center; gap: 8px; font-size: 0.85rem; color: #d3cce3; cursor: pointer; user-select: none; }}
    .pill-track {{
      position: relative; width: 36px; height: 20px;
      background: #444466; border-radius: 10px;
      transition: background 0.2s;
      flex-shrink: 0;
    }}
    .pill-track::after {{
      content: ""; position: absolute;
      top: 3px; left: 3px;
      width: 14px; height: 14px;
      background: #fff; border-radius: 50%;
      transition: transform 0.2s;
    }}
    #hide-regular-edges {{ display: none; }}
    #hide-regular-edges:checked + .pill-track {{ background: #7833ff; }}
    #hide-regular-edges:checked + .pill-track::after {{ transform: translateX(16px); }}
    #stats {{ font-size: 0.8rem; color: #d3cce3; margin-left: auto; }}
    #network {{ flex: 1; min-height: 0; }}
  </style>
</head>
<body>
  <header>
    <h1>Heartland Character Network</h1>
    <select id="season-select">
{options_html}
    </select>
    <label id="regular-toggle">
      <input type="checkbox" id="hide-regular-edges">
      <span class="pill-track"></span>
      Hide regular cast edges
    </label>
    <span id="stats"></span>
  </header>
  <div id="network"></div>
  <script>
    const DATA = {json.dumps(all_data)};

    const PHYSICS_OPTIONS = {{
        solver: "forceAtlas2Based",
        forceAtlas2Based: {{ springLength: 500, springConstant: 0.001, damping: 0.4, centralGravity: 0.01 }},
        stabilization: {{ iterations: 200 }},
        enabled: true,
      }}

    const VIS_OPTIONS = {{
      nodes: {{
        shape: "circularImage",
        font: {{ size: 13, color: "#e0e0e0", face: "system-ui" }},
        borderWidth: 4,
        shadow: {{ enabled: true, size: 6, color: "rgba(0,0,0,0.5)" }},
      }},
      edges: {{
        smooth: {{ type: "continuous" }},
      }},
      physics: PHYSICS_OPTIONS,
      interaction: {{ hover: true, tooltipDelay: 100, hideEdgesOnDrag: true }},
    }};

    const container    = document.getElementById("network");
    const statsEl      = document.getElementById("stats");
    const select       = document.getElementById("season-select");
    const toggleLabel  = document.getElementById("regular-toggle");
    const hideRegularCb = document.getElementById("hide-regular-edges");

    const nodes_ds = new vis.DataSet(DATA["All seasons"].nodes);
    const edges_ds = new vis.DataSet(DATA["All seasons"].edges);
    const network  = new vis.Network(container, {{ nodes: nodes_ds, edges: edges_ds }}, VIS_OPTIONS);

    network.on("stabilizationIterationsDone", () => {{
      network.setOptions({{ physics: PHYSICS_OPTIONS }});
      network.fit();
    }});

    function updateStats(d) {{
      statsEl.textContent = d.nodes.length + " characters \u00b7 " + d.edges.length + " connections";
    }}
    updateStats(DATA["All seasons"]);

    function getVisibleEdges(d) {{
      if (hideRegularCb.checked && d.regulars && d.regulars.length > 0) {{
        const regularSet = new Set(d.regulars);
        return d.edges.filter(e => !regularSet.has(e.from) && !regularSet.has(e.to));
      }}
      return d.edges;
    }}

    select.addEventListener("change", function () {{
      const d = DATA[this.value];
      const isAllSeasons = this.value === "All seasons";
      toggleLabel.style.display = isAllSeasons ? "none" : "flex";
      hideRegularCb.checked = false;
      nodes_ds.clear(); edges_ds.clear();
      nodes_ds.add(d.nodes); edges_ds.add(getVisibleEdges(d));
      network.setOptions({{ physics: PHYSICS_OPTIONS }});
      updateStats(d);
    }});

    hideRegularCb.addEventListener("change", function () {{
      const d = DATA[select.value];
      edges_ds.clear();
      edges_ds.add(getVisibleEdges(d));
    }});
  </script>
</body>
</html>"""

    out = DATA_DIR / "heartland_network.html"
    out.write_text(html, encoding="utf-8")
    port = _ensure_server()
    url = f"http://localhost:{port}/data/heartland_network.html"
    webbrowser.open(url)
    print(f"\nOpened \u2192 {url}")

build_browser_graph()

In [ ]:

import json
from pathlib import Path

nb_path = Path("/Users/serena/projects/heartland-network-graph/heartland_network.ipynb")
nb = json.loads(nb_path.read_text())
code_cells = [c for c in nb['cells'] if c['cell_type'] == 'code']
src = code_cells[-1]['source']
# Check key strings are present
checks = [
    'pill-track' in src,
    '!regularSet.has(e.from) && !regularSet.has(e.to)' in src,
    'display: none' in src,
]
print("pill-track CSS:", checks[0])
print("correct filter logic:", checks[1])
print("toggle hidden by default:", checks[2])
